# Phase 2 — canonical detector baseline (Colab, CUDA)

One training run of the **historical** detector configuration against the **frozen canonical
dataset**, then one evaluation on the canonical test split.

**The only intentional change from the historical run is the dataset.** Architecture,
hyperparameters, augmentation, seed and Ultralytics version are all held fixed. Nothing in this
notebook tunes, restarts, or picks a better result.

This notebook is a thin driver: every non-trivial step is a call into `alpr.*`, which is unit
tested in CI. It never calls `ensure_dataset()` — that rebuilds from Roboflow, and this
experiment must consume the frozen export exactly as it is.

**Run cells in order. If the verification cell in step 4 raises, stop — do not train.**

## 1. Runtime — a CUDA GPU is required

The historical run used `device: '0'`. MPS or CPU would be a second experimental change, so this cell refuses to continue without CUDA.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch, sys, platform

assert torch.cuda.is_available(), "No CUDA GPU. Runtime > Change runtime type > T4 GPU."
GPU_NAME = torch.cuda.get_device_name(0)
print(f"python {sys.version.split()[0]}  torch {torch.__version__}  cuda {torch.version.cuda}")
print(f"gpu    {GPU_NAME}")
if "T4" not in GPU_NAME:
    print(f"NOTE: historical run used a T4; this is a {GPU_NAME}. Record it in provenance.")

## 2. Repository and pinned Ultralytics

`8.4.117` is the version recorded inside the historical `best.pt`. Pinning it keeps `optimizer=auto` resolving the same way.

In [ ]:
REPO   = "https://github.com/fayazhussain2821/Automatic-License-Plate-Recognition.git"
COMMIT = "main"   # pin to a specific SHA for a fully reproducible run

import os

if os.path.isdir("/content/ALPR"):
    !cd /content/ALPR && git fetch --quiet origin && git checkout --quiet $COMMIT
else:
    !git clone --quiet $REPO /content/ALPR && cd /content/ALPR && git checkout --quiet $COMMIT

%cd /content/ALPR
!pip install --quiet -e .
!pip install --quiet "ultralytics==8.4.117"

# An editable install registers through a .pth file, which is only read at
# interpreter startup — a running kernel cannot see it otherwise.
import sys
if "/content/ALPR/src" not in sys.path:
    sys.path.insert(0, "/content/ALPR/src")

import subprocess

import alpr, ultralytics

GIT_COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
assert ultralytics.__version__ == "8.4.117", f"got {ultralytics.__version__}, need 8.4.117"
print(f"alpr {alpr.__version__}   ultralytics {ultralytics.__version__}   commit {GIT_COMMIT[:12]}")

## 3. Bring the canonical dataset in

The canonical export is 3,105 **symlinks** into the SSD, so a plain copy would arrive empty.
Make a dereferenced archive on the Mac first (`-h` follows the links, and reads only — the
frozen dataset is not modified):

```bash
cd /Volumes/MySSD/alpr-data/canonical
tar -czhf /Volumes/MySSD/alpr-data/canonical-colab.tar.gz yolo manifest.jsonl provenance.json
shasum -a 256 /Volumes/MySSD/alpr-data/canonical-colab.tar.gz
```

Upload that archive to Drive, then set `ARCHIVE` below. Expected: **86 MB**, sha256
`cf41f5d945b4268b11d60924f1c55d6966a0c3163e8155b2a68bf9b1fd4c7d95`.

The archive preserves the frozen read-only permissions, so the extracted copy is made writable —
that affects only this temporary Colab copy.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ARCHIVE = "/content/drive/MyDrive/canonical-colab.tar.gz"   # <-- set this
DATA_ROOT = "/content/canonical"

import hashlib, pathlib
print("archive sha256:", hashlib.sha256(pathlib.Path(ARCHIVE).read_bytes()).hexdigest())

!mkdir -p $DATA_ROOT && tar -xzf "$ARCHIVE" -C $DATA_ROOT
# Ultralytics writes <split>.cache beside the labels; the extracted tree
# inherits the frozen read-only bits. Only this temporary copy is touched.
!chmod -R u+w $DATA_ROOT
!ls $DATA_ROOT && ls $DATA_ROOT/yolo

## 4. Verify the frozen dataset — STOP on mismatch

Identity is the two SHA-256 hashes, not the path. `verify_dataset` recomputes them and raises on
any mismatch, so a truncated upload or a rebuilt dataset cannot be trained on by accident.

In [ ]:
from alpr.baseline import inspect_dataset, verify_dataset

EXPORT   = f"{DATA_ROOT}/yolo"
MANIFEST = f"{DATA_ROOT}/manifest.jsonl"

CANONICAL_MANIFEST_SHA = "1058c65db750deb42f3333aba56a3a60d606c26bd0ee793d268cdedb818aeafc"
CANONICAL_EXPORT_SHA   = "8e30e57d4644f6ca4a67db9088d130a79b7f0a0b0f59ad30a9a2460f00cfb983"

facts = inspect_dataset(EXPORT, MANIFEST)
print(facts.summary())

verify_dataset(
    facts,
    export_sha256=CANONICAL_EXPORT_SHA,
    manifest_sha256=CANONICAL_MANIFEST_SHA,
    counts={"train": 2175, "val": 465, "test": 465},
    annotations=3273,
    manifest_records=3105,
)
print("\nVERIFIED — this is the frozen canonical dataset.")

## 5. A `data.yaml` for this location

The canonical `data.yaml` carries the Mac's absolute path and is frozen read-only. This writes a **new** file outside the export; the original is never touched.

In [ ]:
from alpr.baseline import write_data_yaml

DATA_YAML = write_data_yaml(EXPORT, f"{DATA_ROOT}/data_colab.yaml")
print(DATA_YAML.read_text())
print("canonical data.yaml, unchanged:")
print(open(f"{EXPORT}/data.yaml").read())

## 6. The historical configuration, unchanged

Loaded from the committed `configs/detector.yaml`. The assertions below are the contract for this stage — if any fails, the experiment has drifted.

In [ ]:
from alpr.train import TrainConfig

config = TrainConfig.from_yaml("configs/detector.yaml")

expected = dict(model="yolov8s.pt", imgsz=640, epochs=100, batch=16, patience=25,
                seed=0, workers=8, cache=False, device="0",
                degrees=10.0, translate=0.10, scale=0.50, shear=2.0,
                perspective=0.0005, flipud=0.0, fliplr=0.5,
                mosaic=1.0, close_mosaic=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4)
for key, want in expected.items():
    got = getattr(config, key)
    assert got == want, f"{key}: {got!r} != historical {want!r}"

print("historical configuration confirmed:")
for key, want in expected.items():
    print(f"  {key:<12} {want}")
print("\noptimizer stays 'auto' (Ultralytics resolves it; do not replace with AdamW)")

## 7. Train — exactly one run

`train` for fitting, `val` for checkpoint selection (Ultralytics' default `split: val`), `test`
untouched until step 10.

~1–1.5 h on a T4. Do not restart or re-run based on the outcome.

In [ ]:
from datetime import datetime, timezone
from alpr.train import train

STARTED = datetime.now(timezone.utc).isoformat()
results = train(config, DATA_YAML)          # require_gpu=True by default
FINISHED = datetime.now(timezone.utc).isoformat()
print(f"started  {STARTED}\nfinished {FINISHED}")

## 8. Checkpoint and the optimizer `auto` actually chose

The historical run never recorded this. The checkpoint's `train_args` holds the post-resolution values — `warmup_bias_lr` is the tell: Ultralytics forces it to `0.0` when `auto` selects an Adam-family optimizer.

In [ ]:
import torch
from alpr.baseline import sha256_file
from alpr.train import best_weights

WEIGHTS = best_weights(config, results)
CHECKPOINT_SHA = sha256_file(WEIGHTS)
print(f"weights {WEIGHTS}  ({WEIGHTS.stat().st_size/1e6:.1f} MB)")
print(f"sha256  {CHECKPOINT_SHA}")

ckpt = torch.load(WEIGHTS, map_location="cpu", weights_only=False)
ta = ckpt.get("train_args", {})
RESOLVED_OPTIMIZER = {
    "requested": ta.get("optimizer"),
    "warmup_bias_lr_recorded": ta.get("warmup_bias_lr"),
    "lr0_recorded": ta.get("lr0"),
    "momentum_recorded": ta.get("momentum"),
    "adam_family_inferred": ta.get("warmup_bias_lr") == 0.0,
    "note": ("Ultralytics overrides lr0/momentum when optimizer=auto; the optimizer state is "
             "stripped from best.pt, so the family is inferred from warmup_bias_lr, not read directly."),
}
CKPT_META = {"ultralytics_version": ckpt.get("version"), "date": ckpt.get("date")}
print(RESOLVED_OPTIMIZER)
print(CKPT_META)

## 9. Validation metrics from the run

These drove model selection. They are **not** the headline result.

In [ ]:
import csv

VAL_METRICS = {k: float(v) for k, v in (results.results_dict or {}).items()
               if isinstance(v, (int, float))}
print("val (results_dict):", VAL_METRICS)

run_dir = WEIGHTS.parent.parent
rows = list(csv.DictReader(open(run_dir / "results.csv")))
last = rows[-1]
print(f"last epoch {int(float(last['epoch']))}: "
      f"mAP50={float(last['metrics/mAP50(B)']):.4f} "
      f"mAP50-95={float(last['metrics/mAP50-95(B)']):.4f} "
      f"P={float(last['metrics/precision(B)']):.4f} "
      f"R={float(last['metrics/recall(B)']):.4f}")
EPOCHS_RUN = len(rows)
print(f"epochs completed: {EPOCHS_RUN} (patience={config.patience})")

## 10. Final test evaluation — once

The historical run's biggest provenance failure was that its test evaluation left no artifact.
This records the exact settings alongside the numbers.

In [ ]:
from ultralytics import YOLO

EVAL_SETTINGS = {"split": "test", "imgsz": config.imgsz, "conf": None,
                 "iou": 0.7, "max_det": 300, "rect": False, "augment": False,
                 "note": "Ultralytics val() defaults; conf=None means the task default (0.001)."}

model = YOLO(str(WEIGHTS))
m = model.val(data=str(DATA_YAML), split="test", imgsz=config.imgsz, verbose=False)

TEST_METRICS = {"mAP50": float(m.box.map50), "mAP50-95": float(m.box.map),
                "precision": float(m.box.mp), "recall": float(m.box.mr)}
print("CANONICAL TEST:", {k: round(v, 4) for k, v in TEST_METRICS.items()})

## 11. Test metrics by region — EU and IN

Ultralytics reports one number for the split. Region slices come from the project's own
`alpr.evaluate`, which matches greedily by confidence at a fixed threshold — so these are
precision/recall/F1, not mAP, and are not directly comparable to the numbers above.

In [ ]:
from collections import defaultdict
from alpr.data import Split, read_manifest, split_records
from alpr.detect import PlateDetector
from alpr.evaluate import evaluate_records

records = read_manifest(MANIFEST)
test_records = split_records(records, seed=0).partition(records)[Split.TEST]
assert len(test_records) == 465, len(test_records)

by_region = defaultdict(list)
for r in test_records:
    by_region[r.primary_region.value].append(r)
print({k: len(v) for k, v in sorted(by_region.items())})

detector = PlateDetector(WEIGHTS, imgsz=config.imgsz)
REGION_METRICS = {}
for region, recs in sorted(by_region.items()):
    paths = [f"{EXPORT}/images/test/{r.image_id}.jpg" for r in recs]
    preds = []
    for i in range(0, len(paths), 32):
        preds.extend(detector.detect_batch(paths[i:i + 32]))
    rep = evaluate_records(recs, preds)
    REGION_METRICS[region] = {
        "images": len(recs), "ground_truth": rep.overall.ground_truth,
        "precision": round(rep.overall.precision, 4),
        "recall": round(rep.overall.recall, 4),
        "f1": round(rep.overall.f1, 4),
        "true_positives": rep.overall.true_positives,
        "false_positives": rep.overall.false_positives,
        "false_negatives": rep.overall.false_negatives,
    }
    print(f"{region}: {REGION_METRICS[region]}")

REGION_METRICS["_settings"] = {
    "confidence": detector.confidence, "nms_iou": detector.iou_threshold,
    "match_iou": 0.5, "metric": "greedy confidence-ordered matching (alpr.evaluate)",
}

## 12. Provenance

Every field below was measured in this notebook. Unmeasured sections stay null.

In [ ]:
import platform, sys
from dataclasses import asdict
from importlib.metadata import version
from alpr.baseline import baseline_provenance

record = baseline_provenance(
    dataset=facts,
    dataset_paths={"export": EXPORT, "manifest": MANIFEST,
                   "data_yaml": str(DATA_YAML), "source": "frozen canonical, transferred"},
    train_config=asdict(config),
    resolved_optimizer=RESOLVED_OPTIMIZER,
    environment={
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "gpu": GPU_NAME,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "ultralytics": version("ultralytics"),
        "colab": True,
    },
    code={"git_commit": GIT_COMMIT, "alpr_version": alpr.__version__,
          "config_file": "configs/detector.yaml"},
    val_metrics=VAL_METRICS,
    test_metrics=TEST_METRICS,
    region_metrics=REGION_METRICS,
    evaluation=EVAL_SETTINGS,
    checkpoint={"sha256": CHECKPOINT_SHA, "path": str(WEIGHTS),
                "bytes": WEIGHTS.stat().st_size, "start_from": config.model,
                "pretrained": True, **CKPT_META},
    timestamps={"train_started": STARTED, "train_finished": FINISHED,
                "epochs_completed": EPOCHS_RUN},
    notes=[
        "Only intentional change vs the historical run: historical dataset -> canonical dataset.",
        "Historical published result (mAP@50 0.9921 / recall 0.9917) was measured on the "
        "ORIGINAL 2174/466/465 split and its test artifact is unavailable; it is not a "
        "like-for-like comparison and no causal claim is made about any difference.",
        "Region metrics are precision/recall from alpr.evaluate at a fixed confidence, "
        "not mAP, and are not comparable to the Ultralytics numbers.",
    ],
)
out = record.write("/content/baseline_provenance.json")
print(out.read_text()[:1200], "...")

## 13. Bring the results back

Download **artifacts only**. Do not commit weights or the dataset — `.pt` and `data/**` are
gitignored, and the canonical dataset lives on the SSD.

In [ ]:
from google.colab import files

!cp "$run_dir/results.csv" /content/baseline_results.csv
!cp "$run_dir/args.yaml"   /content/baseline_args.yaml

for f in ("/content/baseline_provenance.json", "/content/baseline_results.csv",
          "/content/baseline_args.yaml"):
    files.download(f)

print("Also download the weights separately and keep them OUT of git:")
print(f"  {WEIGHTS}   sha256 {CHECKPOINT_SHA}")